# Imports n initialization

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda, float64, complex128
from numba.cuda import jit as cuda_jit
import math

import few

from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode import KerrEccEqFlux
from few.amplitude.ampinterp2d import AmpInterpKerrEccEq
from few.summation.interpolatedmodesum import InterpolatedModeSum 


from few.utils.ylm import GetYlms

from few import get_file_manager

from few.waveform import GenerateEMRIWaveform, FastKerrEccentricEquatorialFlux

from few.utils.geodesic import get_fundamental_frequencies

from few.utils.constants import YRSID_SI
from smt.sampling_methods import LHS


import os
import sys

# Changing directory to FEWNEW/work
# to import stuffs
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/')
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/')

import GWfuncs
import loglikealt
import modeselectoralt
import parismc
# import gc
import pickle
import cupy as cp

# tune few configuration
cfg_set = few.get_config_setter(reset=True)
cfg_set.set_log_level("info")

# GPU configuration 
use_gpu = True
force_backend = "cuda12x"  
dt = 10     # Time step
T = 1/12     # Total time
print(f"Using dt = {dt} seconds, T = {T} years")

print('Initializing waveform generator...')
# keyword arguments for inspiral generator 
inspiral_kwargs={
        "func": 'KerrEccEqFlux',
        "DENSE_STEPPING": 0, #change to 1/True for uniform sampling
        "include_minus_m": False, 
}

# keyword arguments for inspiral generator 
amplitude_kwargs = {
    "force_backend": force_backend # Force GPU
}

# keyword arguments for Ylm generator (GetYlms)
Ylm_kwargs = {
    "force_backend": force_backend,  # Force GPU
    # "assume_positive_m": True  # if we assume positive m, it will generate negative m for all m>0
}

# keyword arguments for summation generator (InterpolatedModeSum)
sum_kwargs_comb = {
    "force_backend": force_backend,  # Force GPU
    "pad_output": True,
}

sum_kwargs_sep = {
    "force_backend": force_backend,  # Force GPU
    "pad_output": True,
    "separate_modes": True,
}

print("Creating GenerateEMRIWaveform class...")
# Kerr eccentric flux
waveform_gen_comb = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_comb,
    use_gpu=use_gpu
)

# Kerr eccentric flux
waveform_gen_sep = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_sep,
    use_gpu=use_gpu
)


print('Done initializing waveform generator.')

print("Creating GravWaveAnalysis class...")
gwf = GWfuncs.GravWaveAnalysis(T, dt)

print("Initializing loglike class...")


# Source parameters
m1 = 1e6
m2 = 3e1
a = 0.7
p0 = 15
e0 = 0.4 
# NOTE: BELOW FIXED
xI0 = 1.0
dist = 0.25 # Gpc
# Polar and azimuthal angles .. detector frame
# S = Solar system barycenter
# K = spin angular momentum of the MBH
qS = 0.5 
phiS = 1 
qK = 1 #fixed
phiK = phiS + np.pi/3
# Phases
Phi_phi0 = 0.4
Phi_theta0 = 0.0 # equatorial
Phi_r0 = 0.5

params_star = (m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0)
param_true = [np.log10(m1), np.log10(m2), a, p0, e0]

Using dt = 10 seconds, T = 0.08333333333333333 years
Initializing waveform generator...
Creating GenerateEMRIWaveform class...
Done initializing waveform generator.
Creating GravWaveAnalysis class...
Initializing loglike class...


# EMRI Waveforms

In [2]:
# Test parameters
proc1_params = np.array([6.02680358, 1.46982606, 0.7897165, 14.39679547, 0.39827987])
proc2_params = np.array([5.93845955, 1.50506177, 0.45790577, 16.56223858, 0.39802901])

In [3]:
data_waveform = waveform_gen_comb(
    m1, m2, a, p0, e0, xI0, dist,
    qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
    dt=dt, T=T
)


In [4]:
# template 1 (from proc 1) 
m1_1, m2_1, a_1, p0_1, e0_1 = 10**proc1_params[0], 10**proc1_params[1], proc1_params[2], proc1_params[3], proc1_params[4]
template1 = waveform_gen_comb(
    m1_1, m2_1, a_1, p0_1, e0_1, xI0, dist,
    qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
    dt=dt, T=T
)

In [5]:
m1_2, m2_2, a_2, p0_2, e0_2 = 10**proc2_params[0], 10**proc2_params[1], proc2_params[2], proc2_params[3], proc2_params[4]
template2 = waveform_gen_comb(
    m1_2, m2_2, a_2, p0_2, e0_2, xI0, dist,
    qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
    dt=dt, T=T
)

In [6]:
def timemax_inner(d,h):
    # Zero padding
    n = max(len(d), len(h))
    d_pad = np.pad(d, (0, n - len(d)), 'constant')
    h_pad = np.pad(h, (0, n - len(h)), 'constant')

    # fft 
    D = np.fft.fft(d_pad)
    H = np.fft.fft(h_pad)

    # compute Y 
    Y = D * np.conj(H)

    # ifft 
    S = np.fft.ifft(Y)

    # find max corr 
    return np.max(np.abs(S))

In [7]:
from lisatools.sensitivity import get_sensitivity, CornishLISASens
YRSID_SI = 31558149.763545603
def timemax_inner_full(h1, h2, T, dt=10):

    H1 = cp.fft.fft(h1) * dt
    print(H1)

    H2 = cp.fft.fft(h2) * dt
    print(H2)

    N = int(T*YRSID_SI/dt)+1
    fft_freqs = cp.fft.fftfreq(N,dt)
    print(fft_freqs)
    df = 1/(N*dt)
    Sn = get_sensitivity(fft_freqs[1:], sens_fn=CornishLISASens, return_type="PSD")


    Y = cp.zeros_like(H1)

    Y[1:] = H1[1:] * cp.conj(H2[1:]) / (0.5 * Sn)

    S = cp.fft.ifft(Y) / dt
    
    return cp.max(cp.abs(S))

In [8]:
np.sqrt(timemax_inner_full(data_waveform, data_waveform, T))

[1.85148616e-20-8.78342927e-20j 1.85364111e-20-8.79558848e-20j
 1.85581288e-20-8.80777675e-20j ... 1.84512312e-20-8.74712225e-20j
 1.84722707e-20-8.75919652e-20j 1.84934811e-20-8.77129874e-20j]
[1.85148616e-20-8.78342927e-20j 1.85364111e-20-8.79558848e-20j
 1.85581288e-20-8.80777675e-20j ... 1.84512312e-20-8.74712225e-20j
 1.84722707e-20-8.75919652e-20j 1.84934811e-20-8.77129874e-20j]
[ 0.00000000e+00  3.80249824e-07  7.60499648e-07 ... -1.14074947e-06
 -7.60499648e-07 -3.80249824e-07]


array(5.2799268)

In [9]:
np.sqrt(timemax_inner_full(data_waveform, template1, T))

[1.85148616e-20-8.78342927e-20j 1.85364111e-20-8.79558848e-20j
 1.85581288e-20-8.80777675e-20j ... 1.84512312e-20-8.74712225e-20j
 1.84722707e-20-8.75919652e-20j 1.84934811e-20-8.77129874e-20j]
[1.85984460e-20-8.94782187e-20j 1.86203549e-20-8.96016199e-20j
 1.86424358e-20-8.97253121e-20j ... 1.85337611e-20-8.91097235e-20j
 1.85551482e-20-8.92322741e-20j 1.85767102e-20-8.93551046e-20j]
[ 0.00000000e+00  3.80249824e-07  7.60499648e-07 ... -1.14074947e-06
 -7.60499648e-07 -3.80249824e-07]


array(5.30787565)

In [10]:
np.sqrt(timemax_inner_full(template1, template1, T))

[1.85984460e-20-8.94782187e-20j 1.86203549e-20-8.96016199e-20j
 1.86424358e-20-8.97253121e-20j ... 1.85337611e-20-8.91097235e-20j
 1.85551482e-20-8.92322741e-20j 1.85767102e-20-8.93551046e-20j]
[1.85984460e-20-8.94782187e-20j 1.86203549e-20-8.96016199e-20j
 1.86424358e-20-8.97253121e-20j ... 1.85337611e-20-8.91097235e-20j
 1.85551482e-20-8.92322741e-20j 1.85767102e-20-8.93551046e-20j]
[ 0.00000000e+00  3.80249824e-07  7.60499648e-07 ... -1.14074947e-06
 -7.60499648e-07 -3.80249824e-07]


array(5.33917025)

In [13]:
gwf.rhostat_timemax(template2)

array(5.0383843)

In [ ]:
data_waveform

In [ ]:
YRSID_SI = 31558149.763545603
def inner(h1f, h2f, T, dt):
    N = T*YRSID_SI/dt + 1
    print(N)
    df = 1/(N*dt) 
    print(df)

    # Get sensitivity
    fft_freqs = cp.fft.rfftfreq(N, dt)
    print(fft_freqs)

    Sn = get_sensitivity(fft_freqs[1:], sens_fn=CornishLISASens, return_type="PSD")

    plus = cp.conj(h1f[0,1:]) @ (h2f[0,1:] / Sn)
    cross = cp.conj(h1f[1,1:]) @ (h2f[1,1:] / Sn)

    inner_prod = 4*df*(plus+cross)
    print(inner_prod)

    return cp.max(cp.abs(inner_prod))


In [ ]:
data_f = gwf.freq_wave(data_waveform)
print(data_f)
np.sqrt(inner(data_f,data_f, T, dt))

In [ ]:
import cupy as cp  # or numpy as np
timemax_inner_full(data_waveform, data_waveform)


In [ ]:
# <d|d>, <d|h1>, <d|h2> without normalization
timemax_inner(data_waveform, data_waveform), timemax_inner(data_waveform, template1), timemax_inner(data_waveform, template2)

In [ ]:
# <d|d> with normalization
timemax_inner(data_waveform, data_waveform) / (np.sqrt(timemax_inner(data_waveform, data_waveform)) * np.sqrt(timemax_inner(data_waveform, data_waveform)))

In [ ]:
# <d|h1> with normalization
timemax_inner(data_waveform, template1) / (np.sqrt(timemax_inner(data_waveform, data_waveform)) * np.sqrt(timemax_inner(template1, template1)))

In [ ]:
# <d|h2> with normalization
timemax_inner(data_waveform, template2) / (np.sqrt(timemax_inner(data_waveform, data_waveform)) * np.sqrt(timemax_inner(template2, template2)))

In [ ]:
#Test LogLike
loglike_obj = loglikealt.LogLike(params_star, waveform_gen_comb, gwf, 
                                  verbose=True, waveform_gen_sep=waveform_gen_sep,
                                  ell=2, n_vals=np.arange(-5, 0), M_mode=None,
                                  )

print(f"\nSelected modes: {loglike_obj.selected_labels}")

In [ ]:
for group in loglike_obj.selected_labels:
    print(group)

In [ ]:
waveforms_per_group_proc1 = []
for group in loglike_obj.selected_labels:
    waveform_group = waveform_gen_comb(
        m1_1, m2_1, a_1, p0_1, e0_1, xI0, dist,
        qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
        dt=dt, T=T,
        mode_selection=group,
        include_minus_mkn=False)

    waveforms_per_group_proc1.append(waveform_group)


In [ ]:
gwf.rhostat(waveforms_per_group_proc1[0])

In [ ]:
rhostats_proc1 = []
for wf in waveforms_per_group_proc1:
    rhostats_proc1.append(gwf.rhostat(wf).get())

In [ ]:
np.sqrt(np.sum([rho**2 for rho in rhostats_proc1]))


In [ ]:
data_waveform.shape

In [ ]:
waveforms_per_group_proc1[0].shape

In [ ]:
gwf.inner(gwf.freq_wave(data_waveform), gwf.freq_wave(waveforms_per_group_proc1[0]))


In [ ]:
Xm_proc1 = gwf.Xmstat(data_waveform, waveforms_per_group_proc1, cp.asarray(rhostats_proc1))


In [ ]:
chi2_proc1 = gwf.chi_sq(Xm_proc1, cp.asarray(rhostats_proc1))
chi2_proc1

In [ ]:
np.exp(-0.5 * chi2_proc1).get()

In [ ]:
rho_dom_M = np.max(rhostats_proc1)
rho_dom_M 

In [ ]:
rho_tot = gwf.rhostat(data_waveform).get()
rho_tot

In [ ]:
beta = gwf.calc_beta(rho_dom_M, rho_tot)
beta

In [ ]:
np.exp(-0.5 * beta * chi2_proc1).get()

In [ ]:
Xm_full = gwf.Xmstat(data_waveform, [template1], cp.array([gwf.rhostat(template1).get()]))
chi2_full = gwf.chi_sq(Xm_full, cp.array([gwf.rhostat(template1).get()]))
print(f"Chi2 full waveform: {chi2_full.get()}")


In [ ]:
# Time-maximized inner product (PSD-weighted, maximizes over time shifts)
def inner_timemax(d, h, xp=cp):
    """
    Compute noise-weighted inner product maximized over time shifts.
    Uses FFT correlation trick.
    """
    from lisatools.sensitivity import get_sensitivity, CornishLISASens
    
    # Zero padding
    n = max(len(d), len(h))
    d_pad = xp.pad(d, (0, n - len(d)), 'constant')
    h_pad = xp.pad(h, (0, n - len(h)), 'constant')
    
    # Compute FFT frequencies
    fft_freqs = np.fft.rfftfreq(len(d_pad), dt)
    df = fft_freqs[1] - fft_freqs[0]
    
    # Get PSD
    Sn = get_sensitivity(fft_freqs[1:], sens_fn=CornishLISASens, return_type="PSD")
    Sn = xp.asarray(Sn)
    
    # rfft
    D = xp.fft.rfft(d_pad)[1:]
    H = xp.fft.rfft(h_pad)[1:]
    
    # PSD-weighted correlation
    Y = (D * xp.conj(H)) / Sn * (4 * df)
    
    # irfft to get time-domain correlation
    S = xp.fft.irfft(Y)
    
    # Return max (maximizes over time and phase via abs)
    return xp.max(xp.abs(S))

# Time-maximized Xmstat
def Xmstat_timemax(x, hm_arr, rho_modes, xp=cp):
    """
    Calculate X_m statistic with time maximization for each mode
    """
    X_modes = xp.empty(len(hm_arr), dtype=xp.float64)
    
    for idx, hm in enumerate(hm_arr):
        # Time-maximized inner product
        inner_product = inner_timemax(x, hm, xp=xp)
        
        # X_m = <x|hm>_max / rho_m
        X_modes[idx] = inner_product / rho_modes[idx]
    
    return X_modes

In [ ]:
# Add a deliberate time shift to the template
# Shift template1 by some samples
shift_amount = 1000  # samples
template1_shifted = cp.roll(template1, shift_amount)

# Compare chi2 with and without the shift
Xm_shifted = gwf.Xmstat(data_waveform, [template1_shifted], cp.array([gwf.rhostat(template1_shifted).get()]))
chi2_shifted = gwf.chi_sq(Xm_shifted, cp.array([gwf.rhostat(template1_shifted).get()]))

print(f"Chi2 unshifted: {chi2_proc1.get()}")
print(f"Chi2 shifted (no time-max): {chi2_shifted.get()}")

# Now with time maximization it should give same result
Xm_shifted_timemax = Xmstat_timemax(data_waveform, [template1_shifted], cp.array([gwf.rhostat(template1_shifted).get()]))
chi2_shifted_timemax = gwf.chi_sq(Xm_shifted_timemax, cp.array([gwf.rhostat(template1_shifted).get()]))
print(f"Chi2 shifted (with time-max): {chi2_shifted_timemax.get()}")


In [ ]:
# 1. Check if X·ρ = <d|h> (equation 11 from the paper)
# For the full template (not modes)
h_temp_freq = gwf.freq_wave(template1)
standard_inner = gwf.inner(gwf.freq_wave(data_waveform), h_temp_freq)
rho_temp = gwf.rhostat(template1).get()

# Now compute X·ρ from modes
Xm_proc1_array = Xm_proc1.get()
rhostats_proc1_array = np.array(rhostats_proc1)
X_dot_rho = np.sum(Xm_proc1_array * rhostats_proc1_array)

print(f"Standard <d|h>: {standard_inner.get()}")
print(f"X·ρ from modes: {X_dot_rho}")
print(f"Ratio: {X_dot_rho / standard_inner.get()}")
print(f"Should be close to 1.0")

# 2. Check mode orthogonality - modes should have low overlap
print("\nMode overlaps:")
for i in range(len(waveforms_per_group_proc1)):
    for j in range(i+1, len(waveforms_per_group_proc1)):
        overlap = gwf.overlap(gwf.freq_wave(waveforms_per_group_proc1[i]), 
                             gwf.freq_wave(waveforms_per_group_proc1[j]))
        print(f"Mode {i} <-> Mode {j}: {overlap.get():.4f}")


In [ ]:
np.sqrt(gwf.inner(gwf.freq_wave(data_waveform), gwf.freq_wave(template1)))

In [ ]:
def Xstat(self, d, h, maximize_time=False):
    if maximize_time:
        # FFT-based time maximization
        d_f = self.freq_wave(d)
        h_f = self.freq_wave(h)
        correlation_f = d_f * self.xp.conj(h_f) / self.Sn
        correlation_t = self.xp.fft.ifft(correlation_f)
        return self.xp.max(self.xp.abs(correlation_t))
    else:
        # Current: <d|h> at template params
        return self.inner(self.freq_wave(d), self.freq_wave(h))


In [ ]:
def Xstat_timemax(x, h, gwf):
    """
    Compute time-and-phase-maximized detection statistic with PSD weighting.
    
    Returns max_τ,φ |<x|h(τ,φ)>|_weighted
    
    Should peak at sqrt(<h|h>) = SNR when x=h.
    """
    # Use gwf's freq_wave to match other methods (includes dt scaling)
    xf = gwf.freq_wave(x)  # Shape: [2, N_freq] for both polarizations
    hf = gwf.freq_wave(h)
    
    # Frequency resolution
    df = 1.0 / (gwf.N * gwf.dt)
    
    # Get PSD
    Sn = get_sensitivity(gwf.fft_freqs[1:], sens_fn=CornishLISASens, return_type="PSD")
    Sn = gwf.xp.asarray(Sn)
    
    # PSD-weighted correlation for both polarizations
    Y_plus = (xf[0, 1:] * gwf.xp.conj(hf[0, 1:])) / Sn
    Y_cross = (xf[1, 1:] * gwf.xp.conj(hf[1, 1:])) / Sn
    
    # IFFT each polarization
    S_plus = gwf.xp.fft.irfft(Y_plus, n=gwf.N)
    S_cross = gwf.xp.fft.irfft(Y_cross, n=gwf.N)
    
    # Sum polarizations
    S = S_plus + S_cross
    
    # Find max correlation over time and phase
    return gwf.xp.max(gwf.xp.abs(S))

In [ ]:
Xstat_timemax(data_waveform, data_waveform, gwf)

In [ ]:
Xstat_timemax(data_waveform, template1, gwf)

# Modes

In [ ]:
loglike_obj_multi_ell = loglikealt.LogLike(
    params_star, waveform_gen_comb, gwf, 
    verbose=True, 
    waveform_gen_sep=waveform_gen_sep,
    ell=[2, 3],      # Search both ℓ=2 and ℓ=3
    n_vals=np.arange(-3, 2),  # Reasonable n range
    M_mode=5,        # Keep top 5 groups
)

In [ ]:
# Test ℓ=2 alone
loglike_obj_ell2 = loglikealt.LogLike(
    params_star, waveform_gen_comb, gwf, 
    verbose=True, 
    waveform_gen_sep=waveform_gen_sep,
    ell=2,
    n_vals=np.arange(-3, 2),
    M_mode=5,
)

# Test ℓ=3 alone
loglike_obj_ell3 = loglikealt.LogLike(
    params_star, waveform_gen_comb, gwf, 
    verbose=True, 
    waveform_gen_sep=waveform_gen_sep,
    ell=3,
    n_vals=np.arange(-3, 2),
    M_mode=5,
)

print(f"ℓ=2 modes: {loglike_obj_ell2.selected_labels}")
print(f"ℓ=3 modes: {loglike_obj_ell3.selected_labels}")

# Manually combine top modes from both
# Then test SNR capture with mixed selection

In [ ]:
# Test top n-groups with ALL m values included
top_mode_groups = [
    [(3, 0, -5), (3, 1, -5), (3, 2, -5), (3, 3, -5), (3, -1, -5), (3, -2, -5), (3, -3, -5)],  # ℓ=3, n=-5
    [(2, 0, -5), (2, 1, -5), (2, 2, -5), (2, -1, -5), (2, -2, -5)],  # ℓ=2, n=-5
    [(2, 0, -2), (2, 1, -2), (2, 2, -2), (2, -1, -2), (2, -2, -2)],  # ℓ=2, n=-2
    [(2, 0, -1), (2, 1, -1), (2, 2, -1), (2, -1, -1), (2, -2, -1)],  # ℓ=2, n=-1
    [(3, 0, -4), (3, 1, -4), (3, 2, -4), (3, 3, -4), (3, -1, -4), (3, -2, -4), (3, -3, -4)],  # ℓ=3, n=-4
]

wf_groups = []
for group in top_mode_groups:
    wf = waveform_gen_comb(
        m1, m2, a, p0, e0, xI0, dist,
        qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
        dt=dt, T=T, mode_selection=group, include_minus_mkn=False
    )
    wf_groups.append(wf)

rhos_groups = [gwf.rhostat(wf).get() for wf in wf_groups]
print(f"Group SNRs: {rhos_groups}")
rho_quad = np.sqrt(np.sum([r**2 for r in rhos_groups]))
print(f"Total SNR from 5 groups: {rho_quad}")
print(f"Full waveform SNR: {gwf.rhostat(data_waveform).get()}")
print(f"Capture: {rho_quad/gwf.rhostat(data_waveform).get():.2%}")


In [ ]:
# Test with ℓ=3 (which had the highest individual mode SNR) and n ∈ [-5, 0)
loglike_obj_ell3_n5 = loglikealt.LogLike(
    params_star, waveform_gen_comb, gwf, 
    verbose=True, 
    waveform_gen_sep=waveform_gen_sep,
    ell=3,
    n_vals=np.arange(-5, 0),  # n = -5, -4, -3, -2, -1
    M_mode=5,  # Keep all 5 groups
)

print(f"\nSelected modes: {loglike_obj_ell3_n5.selected_labels}")

# Generate waveforms and check SNR capture
wf_ell3 = []
for group in loglike_obj_ell3_n5.selected_labels:
    wf = waveform_gen_comb(
        m1, m2, a, p0, e0, xI0, dist,
        qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
        dt=dt, T=T, mode_selection=group, include_minus_mkn=False
    )
    wf_ell3.append(wf)

rhos_ell3 = [gwf.rhostat(wf).get() for wf in wf_ell3]
rho_quad_ell3 = np.sqrt(np.sum([r**2 for r in rhos_ell3]))
print(f"\nℓ=3 mode SNRs: {rhos_ell3}")
print(f"Total SNR: {rho_quad_ell3}")
print(f"Capture: {rho_quad_ell3/gwf.rhostat(data_waveform).get():.2%}")

In [ ]:
# Test ℓ=2 with n ∈ [-5, 0)
loglike_obj_ell2_n5 = loglikealt.LogLike(
    params_star, waveform_gen_comb, gwf, 
    verbose=True, 
    waveform_gen_sep=waveform_gen_sep,
    ell=2,
    n_vals=np.arange(-5, 0),
    M_mode=5,
)

print(f"\nSelected modes: {loglike_obj_ell2_n5.selected_labels}")

# Generate waveforms and check SNR capture
wf_ell2 = []
for group in loglike_obj_ell2_n5.selected_labels:
    wf = waveform_gen_comb(
        m1, m2, a, p0, e0, xI0, dist,
        qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
        dt=dt, T=T, mode_selection=group, include_minus_mkn=False
    )
    wf_ell2.append(wf)

rhos_ell2 = [gwf.rhostat(wf).get() for wf in wf_ell2]
rho_quad_ell2 = np.sqrt(np.sum([r**2 for r in rhos_ell2]))
print(f"\nℓ=2 mode SNRs: {rhos_ell2}")
print(f"Total SNR: {rho_quad_ell2}")
print(f"Capture: {rho_quad_ell2/gwf.rhostat(data_waveform).get():.2%}")

In [ ]:
# Manual mixed ℓ selection - top 10 groups by SNR
mixed_mode_groups = [
    # Top from ℓ=3
    [(3, 0, -5), (3, 1, -5), (3, 2, -5), (3, 3, -5), (3, -1, -5), (3, -2, -5), (3, -3, -5)],  # 1.65
    # Top from ℓ=2
    [(2, 0, -5), (2, 1, -5), (2, 2, -5), (2, -1, -5), (2, -2, -5)],  # 1.39
    [(2, 0, -2), (2, 1, -2), (2, 2, -2), (2, -1, -2), (2, -2, -2)],  # 1.36
    [(2, 0, -1), (2, 1, -1), (2, 2, -1), (2, -1, -1), (2, -2, -1)],  # 1.32
    [(3, 0, -4), (3, 1, -4), (3, 2, -4), (3, 3, -4), (3, -1, -4), (3, -2, -4), (3, -3, -4)],  # 1.26
    [(2, 0, -4), (2, 1, -4), (2, 2, -4), (2, -1, -4), (2, -2, -4)],  # 1.25
    [(2, 0, -3), (2, 1, -3), (2, 2, -3), (2, -1, -3), (2, -2, -3)],  # 1.25
    [(3, 0, -3), (3, 1, -3), (3, 2, -3), (3, 3, -3), (3, -1, -3), (3, -2, -3), (3, -3, -3)],  # 0.83
    [(3, 0, -2), (3, 1, -2), (3, 2, -2), (3, 3, -2), (3, -1, -2), (3, -2, -2), (3, -3, -2)],  # 0.52
    [(2, 0, 0), (2, 1, 0), (2, 2, 0), (2, -1, 0), (2, -2, 0)],  # 0.77
]

wf_mixed_10 = []
for group in mixed_mode_groups:
    wf = waveform_gen_comb(
        m1, m2, a, p0, e0, xI0, dist,
        qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
        dt=dt, T=T, mode_selection=group, include_minus_mkn=False
    )
    wf_mixed_10.append(wf)

rhos_mixed_10 = [gwf.rhostat(wf).get() for wf in wf_mixed_10]
rho_quad_mixed_10 = np.sqrt(np.sum([r**2 for r in rhos_mixed_10]))

print(f"Mixed ℓ=2,3 mode SNRs (10 groups): {rhos_mixed_10}")
print(f"Total SNR: {rho_quad_mixed_10}")
print(f"Full waveform SNR: {gwf.rhostat(data_waveform).get()}")
print(f"Capture: {rho_quad_mixed_10/gwf.rhostat(data_waveform).get():.2%}")

In [ ]:
# Combine into 5 mega-groups
mega_mode_groups = [
    # Group 1: All ℓ=3, n=-5 AND ℓ=2, n=-5
    [(3, 0, -5), (3, 1, -5), (3, 2, -5), (3, 3, -5), (3, -1, -5), (3, -2, -5), (3, -3, -5),
     (2, 0, -5), (2, 1, -5), (2, 2, -5), (2, -1, -5), (2, -2, -5)],
    
    # Group 2: All ℓ=3, n=-4 AND ℓ=2, n=-4
    [(3, 0, -4), (3, 1, -4), (3, 2, -4), (3, 3, -4), (3, -1, -4), (3, -2, -4), (3, -3, -4),
     (2, 0, -4), (2, 1, -4), (2, 2, -4), (2, -1, -4), (2, -2, -4)],
    
    # Group 3: All ℓ=3, n=-3 AND ℓ=2, n=-3
    [(3, 0, -3), (3, 1, -3), (3, 2, -3), (3, 3, -3), (3, -1, -3), (3, -2, -3), (3, -3, -3),
     (2, 0, -3), (2, 1, -3), (2, 2, -3), (2, -1, -3), (2, -2, -3)],
    
    # Group 4: All ℓ=3, n=-2 AND ℓ=2, n=-2
    [(3, 0, -2), (3, 1, -2), (3, 2, -2), (3, 3, -2), (3, -1, -2), (3, -2, -2), (3, -3, -2),
     (2, 0, -2), (2, 1, -2), (2, 2, -2), (2, -1, -2), (2, -2, -2)],
    
    # Group 5: All ℓ=3, n=-1 AND ℓ=2, n=-1
    [(3, 0, -1), (3, 1, -1), (3, 2, -1), (3, 3, -1), (3, -1, -1), (3, -2, -1), (3, -3, -1),
     (2, 0, -1), (2, 1, -1), (2, 2, -1), (2, -1, -1), (2, -2, -1)],
]

wf_mega = []
for group in mega_mode_groups:
    wf = waveform_gen_comb(
        m1, m2, a, p0, e0, xI0, dist,
        qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
        dt=dt, T=T, mode_selection=group, include_minus_mkn=False
    )
    wf_mega.append(wf)

rhos_mega = [gwf.rhostat(wf).get() for wf in wf_mega]
rho_quad_mega = np.sqrt(np.sum([r**2 for r in rhos_mega]))

print(f"Mega-group SNRs (5 groups): {rhos_mega}")
print(f"Total SNR: {rho_quad_mega}")
print(f"Full waveform SNR: {gwf.rhostat(data_waveform).get()}")
print(f"Capture: {rho_quad_mega/gwf.rhostat(data_waveform).get():.2%}")

In [ ]:
# Final push - add ℓ=6 to group 1 or extend n range
final_mega_groups = [
    # Group 1: n=-7,-6,-5 with ℓ=2,3,4,5,6
    [(2,0,-7),(2,1,-7),(2,2,-7),(2,-1,-7),(2,-2,-7),
     (3,0,-7),(3,1,-7),(3,2,-7),(3,3,-7),(3,-1,-7),(3,-2,-7),(3,-3,-7),
     (4,0,-7),(4,1,-7),(4,2,-7),(4,3,-7),(4,4,-7),(4,-1,-7),(4,-2,-7),(4,-3,-7),(4,-4,-7),
     (5,0,-7),(5,1,-7),(5,2,-7),(5,3,-7),(5,4,-7),(5,5,-7),(5,-1,-7),(5,-2,-7),(5,-3,-7),(5,-4,-7),(5,-5,-7),
     (2,0,-6),(2,1,-6),(2,2,-6),(2,-1,-6),(2,-2,-6),
     (3,0,-6),(3,1,-6),(3,2,-6),(3,3,-6),(3,-1,-6),(3,-2,-6),(3,-3,-6),
     (4,0,-6),(4,1,-6),(4,2,-6),(4,3,-6),(4,4,-6),(4,-1,-6),(4,-2,-6),(4,-3,-6),(4,-4,-6),
     (5,0,-6),(5,1,-6),(5,2,-6),(5,3,-6),(5,4,-6),(5,5,-6),(5,-1,-6),(5,-2,-6),(5,-3,-6),(5,-4,-6),(5,-5,-6),
     (2,0,-5),(2,1,-5),(2,2,-5),(2,-1,-5),(2,-2,-5),
     (3,0,-5),(3,1,-5),(3,2,-5),(3,3,-5),(3,-1,-5),(3,-2,-5),(3,-3,-5),
     (4,0,-5),(4,1,-5),(4,2,-5),(4,3,-5),(4,4,-5),(4,-1,-5),(4,-2,-5),(4,-3,-5),(4,-4,-5),
     (5,0,-5),(5,1,-5),(5,2,-5),(5,3,-5),(5,4,-5),(5,5,-5),(5,-1,-5),(5,-2,-5),(5,-3,-5),(5,-4,-5),(5,-5,-5)],
    
    # Group 2-5: same as ultimate_groups
    ultimate_groups[1],
    ultimate_groups[2],
    ultimate_groups[3],
    ultimate_groups[4],
]

wf_final = []
for i, group in enumerate(final_mega_groups):
    try:
        wf = waveform_gen_comb(
            m1, m2, a, p0, e0, xI0, dist,
            qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
            dt=dt, T=T, mode_selection=group, include_minus_mkn=False
        )
        wf_final.append(wf)
    except Exception as e:
        print(f"Failed group {i}: {e}")

rhos_final = [gwf.rhostat(wf).get() for wf in wf_final]
rho_quad_final = np.sqrt(np.sum([r**2 for r in rhos_final]))

print(f"Final mega-group SNRs: {rhos_final}")
print(f"Total SNR: {rho_quad_final}")
print(f"Capture: {rho_quad_final/gwf.rhostat(data_waveform).get():.2%}")

In [ ]:
# Define your optimal 5 mega-groups that achieve >90% SNR
optimal_mode_groups = [
    # Group 1: n=-7,-6,-5 with ℓ=2,3,4,5
    [(2,0,-7),(2,1,-7),(2,2,-7),(2,-1,-7),(2,-2,-7),
     (3,0,-7),(3,1,-7),(3,2,-7),(3,3,-7),(3,-1,-7),(3,-2,-7),(3,-3,-7),
     (4,0,-7),(4,1,-7),(4,2,-7),(4,3,-7),(4,4,-7),(4,-1,-7),(4,-2,-7),(4,-3,-7),(4,-4,-7),
     (5,0,-7),(5,1,-7),(5,2,-7),(5,3,-7),(5,4,-7),(5,5,-7),(5,-1,-7),(5,-2,-7),(5,-3,-7),(5,-4,-7),(5,-5,-7),
     (2,0,-6),(2,1,-6),(2,2,-6),(2,-1,-6),(2,-2,-6),
     (3,0,-6),(3,1,-6),(3,2,-6),(3,3,-6),(3,-1,-6),(3,-2,-6),(3,-3,-6),
     (4,0,-6),(4,1,-6),(4,2,-6),(4,3,-6),(4,4,-6),(4,-1,-6),(4,-2,-6),(4,-3,-6),(4,-4,-6),
     (5,0,-6),(5,1,-6),(5,2,-6),(5,3,-6),(5,4,-6),(5,5,-6),(5,-1,-6),(5,-2,-6),(5,-3,-6),(5,-4,-6),(5,-5,-6),
     (2,0,-5),(2,1,-5),(2,2,-5),(2,-1,-5),(2,-2,-5),
     (3,0,-5),(3,1,-5),(3,2,-5),(3,3,-5),(3,-1,-5),(3,-2,-5),(3,-3,-5),
     (4,0,-5),(4,1,-5),(4,2,-5),(4,3,-5),(4,4,-5),(4,-1,-5),(4,-2,-5),(4,-3,-5),(4,-4,-5),
     (5,0,-5),(5,1,-5),(5,2,-5),(5,3,-5),(5,4,-5),(5,5,-5),(5,-1,-5),(5,-2,-5),(5,-3,-5),(5,-4,-5),(5,-5,-5)],
    
    # Group 2: n=-4 with ℓ=2,3,4,5
    [(2,0,-4),(2,1,-4),(2,2,-4),(2,-1,-4),(2,-2,-4),
     (3,0,-4),(3,1,-4),(3,2,-4),(3,3,-4),(3,-1,-4),(3,-2,-4),(3,-3,-4),
     (4,0,-4),(4,1,-4),(4,2,-4),(4,3,-4),(4,4,-4),(4,-1,-4),(4,-2,-4),(4,-3,-4),(4,-4,-4),
     (5,0,-4),(5,1,-4),(5,2,-4),(5,3,-4),(5,4,-4),(5,5,-4),(5,-1,-4),(5,-2,-4),(5,-3,-4),(5,-4,-4),(5,-5,-4)],
    
    # Group 3: n=-3 with ℓ=2,3,4,5
    [(2,0,-3),(2,1,-3),(2,2,-3),(2,-1,-3),(2,-2,-3),
     (3,0,-3),(3,1,-3),(3,2,-3),(3,3,-3),(3,-1,-3),(3,-2,-3),(3,-3,-3),
     (4,0,-3),(4,1,-3),(4,2,-3),(4,3,-3),(4,4,-3),(4,-1,-3),(4,-2,-3),(4,-3,-3),(4,-4,-3),
     (5,0,-3),(5,1,-3),(5,2,-3),(5,3,-3),(5,4,-3),(5,5,-3),(5,-1,-3),(5,-2,-3),(5,-3,-3),(5,-4,-3),(5,-5,-3)],
    
    # Group 4: n=-2 with ℓ=2,3,4,5
    [(2,0,-2),(2,1,-2),(2,2,-2),(2,-1,-2),(2,-2,-2),
     (3,0,-2),(3,1,-2),(3,2,-2),(3,3,-2),(3,-1,-2),(3,-2,-2),(3,-3,-2),
     (4,0,-2),(4,1,-2),(4,2,-2),(4,3,-2),(4,4,-2),(4,-1,-2),(4,-2,-2),(4,-3,-2),(4,-4,-2),
     (5,0,-2),(5,1,-2),(5,2,-2),(5,3,-2),(5,4,-2),(5,5,-2),(5,-1,-2),(5,-2,-2),(5,-3,-2),(5,-4,-2),(5,-5,-2)],
    
    # Group 5: n=-1,0,1 with ℓ=2,3,4,5
    [(2,0,-1),(2,1,-1),(2,2,-1),(2,-1,-1),(2,-2,-1),
     (3,0,-1),(3,1,-1),(3,2,-1),(3,3,-1),(3,-1,-1),(3,-2,-1),(3,-3,-1),
     (4,0,-1),(4,1,-1),(4,2,-1),(4,3,-1),(4,4,-1),(4,-1,-1),(4,-2,-1),(4,-3,-1),(4,-4,-1),
     (5,0,-1),(5,1,-1),(5,2,-1),(5,3,-1),(5,4,-1),(5,5,-1),(5,-1,-1),(5,-2,-1),(5,-3,-1),(5,-4,-1),(5,-5,-1),
     (2,0,0),(2,1,0),(2,2,0),(2,-1,0),(2,-2,0),
     (3,0,0),(3,1,0),(3,2,0),(3,3,0),(3,-1,0),(3,-2,0),(3,-3,0),
     (4,0,0),(4,1,0),(4,2,0),(4,3,0),(4,4,0),(4,-1,0),(4,-2,0),(4,-3,0),(4,-4,0),
     (5,0,0),(5,1,0),(5,2,0),(5,3,0),(5,4,0),(5,5,0),(5,-1,0),(5,-2,0),(5,-3,0),(5,-4,0),(5,-5,0),
     (2,0,1),(2,1,1),(2,2,1),(2,-1,1),(2,-2,1),
     (3,0,1),(3,1,1),(3,2,1),(3,3,1),(3,-1,1),(3,-2,1),(3,-3,1)],
]

# Now use these in LogLike with mode_select parameter
loglike_optimal = loglikealt.LogLike(
    params_star, 
    waveform_gen_comb, 
    gwf, 
    verbose=True, 
    waveform_gen_sep=waveform_gen_sep,
    mode_select=optimal_mode_groups,  # Pass pre-defined modes
)

print(f"Using optimal mode groups with {len(optimal_mode_groups)} groups")
print("This should achieve >90% SNR capture and make F-statistic work properly!")


In [ ]:
# Test SNR capture for optimal_mode_groups
import warnings
warnings.filterwarnings('ignore', message='.*Mode selection is large.*')

wf_optimal = []
for i, group in enumerate(optimal_mode_groups):
    wf = waveform_gen_comb(
        m1, m2, a, p0, e0, xI0, dist,
        qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
        dt=dt, T=T, mode_selection=group, include_minus_mkn=False
    )
    wf_optimal.append(wf)

rhos_optimal = [gwf.rhostat(wf).get() for wf in wf_optimal]
rho_quad_optimal = np.sqrt(np.sum([r**2 for r in rhos_optimal]))

print(f"Optimal mode group SNRs: {rhos_optimal}")
print(f"Total SNR: {rho_quad_optimal}")
print(f"Full waveform SNR: {gwf.rhostat(data_waveform).get()}")
print(f"Capture: {rho_quad_optimal/gwf.rhostat(data_waveform).get():.2%}")

In [ ]:
import time

# Time the SNR capture test
start_time = time.time()

wf_optimal = []
for i, group in enumerate(optimal_mode_groups):
    group_start = time.time()
    wf = waveform_gen_comb(
        m1, m2, a, p0, e0, xI0, dist,
        qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,
        dt=dt, T=T, mode_selection=group, include_minus_mkn=False
    )
    wf_optimal.append(wf)
    group_time = time.time() - group_start
    print(f"Group {i+1}: {group_time:.2f}s")

rhos_optimal = [gwf.rhostat(wf).get() for wf in wf_optimal]
rho_quad_optimal = np.sqrt(np.sum([r**2 for r in rhos_optimal]))

total_time = time.time() - start_time

print(f"\nOptimal mode group SNRs: {rhos_optimal}")
print(f"Total SNR: {rho_quad_optimal}")
print(f"Full waveform SNR: {gwf.rhostat(data_waveform).get()}")
print(f"Capture: {rho_quad_optimal/gwf.rhostat(data_waveform).get():.2%}")
print(f"\nTotal runtime: {total_time:.2f}s ({total_time/60:.2f} minutes)")
print(f"Average per group: {total_time/5:.2f}s")


In [ ]:
# Test with optimal modes
loglike_optimal = loglikealt.LogLike(
    params_star, 
    waveform_gen_comb, 
    gwf, 
    verbose=True, 
    waveform_gen_sep=waveform_gen_sep,
    mode_select=optimal_mode_groups,
)

# Test at proc1 parameters
logl_proc1 = loglike_optimal(
    (10**proc1_params[0], 10**proc1_params[1], proc1_params[2], proc1_params[3], proc1_params[4],
     xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0)
)

# Test at wrong parameters
logl_wrong = loglike_optimal(
    (10**6.5, 10**1.8, 0.3, 20.0, 0.2,
     xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0)
)

print(f"Log-likelihood at proc1: {logl_proc1}")
print(f"Log-likelihood at wrong params: {logl_wrong}")
print(f"Difference: {logl_proc1 - logl_wrong}")


# Test

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda, float64, complex128
from numba.cuda import jit as cuda_jit
import math

import few

from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode import KerrEccEqFlux
from few.amplitude.ampinterp2d import AmpInterpKerrEccEq
from few.summation.interpolatedmodesum import InterpolatedModeSum 


from few.utils.ylm import GetYlms

from few import get_file_manager

from few.waveform import GenerateEMRIWaveform, FastKerrEccentricEquatorialFlux

from few.utils.geodesic import get_fundamental_frequencies

from few.utils.constants import YRSID_SI
from smt.sampling_methods import LHS


import os
import sys

# Changing directory to FEWNEW/work
# to import stuffs
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/')
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/')

import GWfuncs
import loglikealt
import modeselectoralt
import parismc
# import gc
import pickle
import cupy as cp

# tune few configuration
cfg_set = few.get_config_setter(reset=True)
cfg_set.set_log_level("info")

# GPU configuration 
use_gpu = True
force_backend = "cuda12x"  
dt = 10     # Time step
T = 1/12     # Total time
print(f"Using dt = {dt} seconds, T = {T} years")

print('Initializing waveform generator...')
# keyword arguments for inspiral generator 
inspiral_kwargs={
        "func": 'KerrEccEqFlux',
        "DENSE_STEPPING": 0, #change to 1/True for uniform sampling
        "include_minus_m": False, 
}

# keyword arguments for inspiral generator 
amplitude_kwargs = {
    "force_backend": force_backend # Force GPU
}

# keyword arguments for Ylm generator (GetYlms)
Ylm_kwargs = {
    "force_backend": force_backend,  # Force GPU
    # "assume_positive_m": True  # if we assume positive m, it will generate negative m for all m>0
}

# keyword arguments for summation generator (InterpolatedModeSum)
sum_kwargs_comb = {
    "force_backend": force_backend,  # Force GPU
    "pad_output": True,
}

sum_kwargs_sep = {
    "force_backend": force_backend,  # Force GPU
    "pad_output": True,
    "separate_modes": True,
}

print("Creating GenerateEMRIWaveform class...")
# Kerr eccentric flux
waveform_gen_comb = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_comb,
    use_gpu=use_gpu
)

# Kerr eccentric flux
waveform_gen_sep = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_sep,
    use_gpu=use_gpu
)


print('Done initializing waveform generator.')

print("Creating GravWaveAnalysis class...")
gwf = GWfuncs.GravWaveAnalysis(T, dt)

print("Initializing loglike class...")


# Source parameters
m1 = 1e6
m2 = 3e1
a = 0.7
p0 = 15
e0 = 0.4 
# NOTE: BELOW FIXED
xI0 = 1.0
dist = 0.25 # Gpc
# Polar and azimuthal angles .. detector frame
# S = Solar system barycenter
# K = spin angular momentum of the MBH
qS = 0.5 
phiS = 1 
qK = 1 #fixed
phiK = phiS + np.pi/3
# Phases
Phi_phi0 = 0.4
Phi_theta0 = 0.0 # equatorial
Phi_r0 = 0.5

params_star = (m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0)
param_true = [np.log10(m1), np.log10(m2), a, p0, e0]

# Explicit mode groups: ℓ∈{2,3}, n∈{-5,-4,-3,-2,-1}
# Group modes by n value, including all m for each (ℓ,n) pair
mega_mode_groups = [
    # Group 1: All ℓ=3, n=-5 AND ℓ=2, n=-5
    [(3, 0, -5), (3, 1, -5), (3, 2, -5), (3, 3, -5), (3, -1, -5), (3, -2, -5), (3, -3, -5),
     (2, 0, -5), (2, 1, -5), (2, 2, -5), (2, -1, -5), (2, -2, -5)],

    # Group 2: All ℓ=3, n=-4 AND ℓ=2, n=-4
    [(3, 0, -4), (3, 1, -4), (3, 2, -4), (3, 3, -4), (3, -1, -4), (3, -2, -4), (3, -3, -4),
     (2, 0, -4), (2, 1, -4), (2, 2, -4), (2, -1, -4), (2, -2, -4)],

    # Group 3: All ℓ=3, n=-3 AND ℓ=2, n=-3
    [(3, 0, -3), (3, 1, -3), (3, 2, -3), (3, 3, -3), (3, -1, -3), (3, -2, -3), (3, -3, -3),
     (2, 0, -3), (2, 1, -3), (2, 2, -3), (2, -1, -3), (2, -2, -3)],

    # Group 4: All ℓ=3, n=-2 AND ℓ=2, n=-2
    [(3, 0, -2), (3, 1, -2), (3, 2, -2), (3, 3, -2), (3, -1, -2), (3, -2, -2), (3, -3, -2),
     (2, 0, -2), (2, 1, -2), (2, 2, -2), (2, -1, -2), (2, -2, -2)],

    # Group 5: All ℓ=3, n=-1 AND ℓ=2, n=-1
    [(3, 0, -1), (3, 1, -1), (3, 2, -1), (3, 3, -1), (3, -1, -1), (3, -2, -1), (3, -3, -1),
     (2, 0, -1), (2, 1, -1), (2, 2, -1), (2, -1, -1), (2, -2, -1)],
]

# OLD n-indexed mode selection parameters (commented out)
# n_vals = np.arange(-5, 0)  # n from -5 to -1
# ell = 2  # quadrupole only

# NOTE: change verbose argument for debugging
# Using explicit mode groups via mode_select parameter
loglike_obj = loglikealt.LogLike(
    params_star,
    waveform_gen_comb,
    gwf,
    verbose=False,
    waveform_gen_sep=waveform_gen_sep,
    mode_select=mega_mode_groups,  # Use explicit mode groups
    M_mode=None  # No SNR filtering, use all mode groups
    # OLD: ell=ell, n_vals=n_vals
)

print('Done initializing loglike class.')
print('Calculating SNR...')
data = loglike_obj.signal
data_snr = gwf.rhostat(data)
print('SNR calculated:', data_snr)

In [ ]:
loglike_obj(np.array([m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0]))
